# Reproducing Heeringa (2024) from dandiset 001262

Paper. Heeringa AN (2024). Single-unit data for sensory neuroscience: Responses from the
auditory nerve of young-adult and aging gerbils. *Scientific Data* 11, 411.
[doi:10.1038/s41597-024-03259-3](https://doi.org/10.1038/s41597-024-03259-3). The claims file
was written from the version of record (open access, PMC11035566), which has no supplement, and
from the paper's MATLAB toolbox on Zenodo (doi:10.5281/zenodo.10370064).

Dandiset. [DANDI:001262](https://dandiarchive.org/dandiset/001262/0.241205.0959), version
0.241205.0959. The paper cites no DANDI version: its data statement points to the Dryad deposit
(doi:10.5061/dryad.qv9s4mwn4), which the dandiset lists as a variant form. Two versions were
published on 2024-12-05, 30 minutes apart; this is the more recent. The data check below shows
that both hold the same 1160 files (same blobs), so the choice changes no number; the later one
adds metadata only (the paper as `relatedResource` and `protocol`, keywords, funder, ethics).

What the dandiset holds. One NWB file per single unit, 1160 units from 104 gerbils, with per
trial the spike times, the stimulus (frequency, level, delay, duration), the raw voltage trace,
and per recording type one row of analysed outcomes (best frequency, spontaneous rate,
threshold). The other analysed outcomes the paper describes (characteristic frequency, vector
strength and its p-value, click latencies) are not in the files, so they are recomputed here from
the spike times with the paper's own functions (named in its Methods), as described in the helper
`gerbil_an.py`. Raw voltage traces are not read.

Outside inputs. None. Method details come from the paper's Methods and from its code
(Zenodo record 10370064), as cited in the helper and under each claim.

| Claim | Tested |
|---|---|
| table1-animals-units | tested |
| table1-age-abr-weight | tested |
| table1-bf-range | tested |
| fig3-abr-age | tested |
| females-fewer-in-old | tested |
| oxygen-78 | tested |
| complex-stimuli-experiments | tested |
| recording-type-counts | tested |
| fig7-sampling-by-age | tested |
| isi-exclusion | tested |
| no-prepotentials | blocked (methods underspecified) |
| rlf-monotonic | blocked (methods underspecified); description shown |
| fig5b-example | tested |
| fig6a-young-two-sensitive-regions | tested |
| fig6b-young-bf-gap | tested |
| fig6c-sr-distributions | tested |
| fig6d-max-phase-locking-bf | tested |
| fig6d-highest-vs-low-bf-low-sr | tested |
| fig7a-cf-matches-bf | tested |
| fig7b-click-latency-bf | tested |
| fig7ab-representative-sampling | tested |
| old-success-rate | blocked (data missing) |
| psth-primary-like | blocked (methods underspecified) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import gerbil_an as g

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = g.list_assets()
units = g.load_all(assets)
print(f"read {len(units)} files of dandiset {g.DANDISET_ID} version {g.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

One row per unit, with the animal metadata parsed from the file and the analysed outcomes from
its analysis table. Each check of the metadata against the values is stated under its output.

In [ ]:
earlier = g.list_assets("0.241205.0929")
same = (set(zip(assets.path, assets.blob)) == set(zip(earlier.path, earlier.blob)))
print(f"Versions 0.241205.0929 and {g.DANDISET_VERSION}: {len(earlier)} and {len(assets)} files; "
      f"same paths and blobs: {same}")
meta = requests.get(
    f"{g.API}/dandisets/{g.DANDISET_ID}/versions/{g.DANDISET_VERSION}/", timeout=60).json()
summ = meta["assetsSummary"]
print("Archive summary: subjects", summ["numberOfSubjects"], "; files", summ["numberOfFiles"],
      "; species", [s["name"] for s in summ["species"]],
      "; variableMeasured", summ["variableMeasured"],
      "; measurementTechnique", [m["name"] for m in summ["measurementTechnique"]])

In [ ]:
rows = []
for u in units:
    a = u["analysis"].set_index("experiment")
    sr_best, sr_source = np.nan, None
    for k in ["SR", "PH", "RLF", "BF"]:          # best estimate of SR, check_dataset_units.m
        if k in a.index and not np.isnan(a.sr[k]):
            sr_best, sr_source = a.sr[k], k
            break
    t = u["trials"]
    rows.append({
        "path": u["path"], "animal": u["subject_id"], "unit": u["identifier"].split(":", 1)[1],
        "age_days": g.age_days(u["age"]), "sex": u["sex"], "weight_g": g.weight_g(u["weight"]),
        "abr_field": g.abr_field(u["notes"]), "oxygen": g.oxygen(u["surgery"]),
        "speaker": g.speaker(u["stimulus_description"]),
        "types": tuple(sorted(t.experiment.unique())),
        "analysis_rows": tuple(u["analysis"].experiment),
        "bf": a.bf.get("BF", np.nan), "rlf_threshold": a.threshold.get("RLF", np.nan),
        "sr_best": sr_best, "sr_source": sr_source, "sr_SR": a.sr.get("SR", np.nan),
        "n_trials": len(t), "n_spikes": int(t.n_spikes.sum()), "nan_placeholders": u["n_nan_spikes"],
        "tags_match": bool(all(tag.startswith(e + "_") for tag, e in zip(t.tag, t.experiment))),
    })
U = pd.DataFrame(rows)
U["abr"] = U.abr_field.map(g.abr_decoded)
U["group"] = U.age_days.map(g.age_group)
by_path = {u["path"]: u for u in units}
print("Units:", len(U), "; animals:", U.animal.nunique(), "; species in files:",
      sorted({u["species"] for u in units}))
print("Each file's subject folder matches its subject_id:",
      bool((U.path.str.split("/").str[0] == "sub-" + U.animal).all()))
print("Animal metadata constant within each animal (age, sex, weight, ABR, oxygen, speaker):",
      bool((U.groupby("animal")[["age_days", "sex", "weight_g", "abr_field", "oxygen", "speaker"]]
            .nunique(dropna=False) <= 1).all().all()))
print("Every trial's tag starts with its recording type:", bool(U.tags_match.all()))
print("Recording types in the analysis table and in the trials agree:",
      bool((U.analysis_rows.map(lambda x: tuple(sorted(x))) == U.types).all()),
      "; analysis rows with a repeated type:", int(U.analysis_rows.map(lambda x: len(x) != len(set(x))).sum()))
print("Units per recording type:")
print(pd.Series([t for ts in U.types for t in ts]).value_counts().to_string())

Every file opens, every subject folder matches the file's `subject_id`, and the animal metadata
are the same in all files of an animal, so animals are counted by `subject_id`. The archive's
summary of 104 subjects matches. Each unit is in one file, so no unit has to be matched across
files. The trial table and the spike-time table have one row per trial in the same order (every
tag begins with the trial's recording type), and each recording type appears once per unit in
the analysis table.

In [ ]:
print("Ages (days), by group:")
print(U.groupby("group").age_days.agg(["min", "max", "size"]).to_string())
print("\nRaw weight strings, examples:", U.drop_duplicates("animal").weight_g.head(3).tolist(),
      [by_path[p]["weight"] for p in U.drop_duplicates("animal").path.head(3)])
print("Weights (g): min", U.weight_g.min(), "max", U.weight_g.max(), "; unparsed:", int(U.weight_g.isna().sum()))
print("Sex values:", U.drop_duplicates("animal").sex.value_counts().to_dict())
print("Oxygen field per animal:", U.drop_duplicates("animal").oxygen.value_counts().to_dict())
print("Speaker per animal:", U.drop_duplicates("animal").speaker.value_counts().to_dict())
print("\nNotes field, example:", repr(units[0]["notes"]))
ab = U.drop_duplicates("animal")
print("\nABR field after 'threshold (db SPL): ' per animal (repr), with its code point:")
print(ab.abr_field.map(lambda s: (repr(s), ord(s) if len(s) == 1 else None)).value_counts().to_string())
dec = ab.abr.dropna()
print("Decoded thresholds: values", sorted(dec.unique().tolist()), "; all multiples of 5:",
      bool((dec % 5 == 0).all()), "; animals with 'NaN':", int((ab.abr_field == "NaN").sum()))

The ABR threshold is not readable as written. In every file `general/notes` ends with
"threshold (db SPL): " followed by a single character, or by "NaN". Read as code points, the
characters are 10 to 85 in steps of 5 (for example "#" is 35 and "K" is 75; some are control
characters such as a line feed, 10), which is the range Table 1 gives (10 to 85 dB SPL). The
number appears to have been written as a character in the conversion to NWB. This is a format
error in the files; the claims that use the ABR threshold read it as the character's code point,
an interpretation the data support (the values are multiples of 5 and in the paper's range) but
do not prove. 11 animals have "NaN". Ages are given in days (`P85D`), weights as text with a unit
(`65g`, `6.110000e+01g`), and oxygen as "yes" or "no" in `general/surgery`.

In [ ]:
print("BF (stored, Hz): min", U.bf.min(), "max", U.bf.max(), "; units with a BF recording:",
      int(U.types.map(lambda x: "BF" in x).sum()), "; BF NaN:", int(U.bf.isna().sum()))
print("RLF threshold (stored, dB SPL): min", round(U.rlf_threshold.min(), 1), "max", round(U.rlf_threshold.max(), 1),
      "; non-NaN", int(U.rlf_threshold.notna().sum()), "of", int(U.types.map(lambda x: "RLF" in x).sum()), "RLF units")
print("SR best estimate (spikes/s): min", U.sr_best.min(), "max", round(U.sr_best.max(), 1),
      "; source:", U.sr_source.value_counts(dropna=False).to_dict())
print("Spike-time NaN placeholders (trials with no spike):", int(U.nan_placeholders.sum()),
      "; trials:", int(U.n_trials.sum()))
allt = pd.concat([u["trials"].assign(path=u["path"]) for u in units], ignore_index=True)
print("\nTrial parameters by recording type:")
summary = allt.groupby("experiment").agg(
    trials=("tag", "size"), delay_ms=("delay", lambda x: sorted(set(np.round(x.dropna() * 1000, 2)))),
    duration_ms=("duration", lambda x: sorted(set(np.round(x.dropna() * 1000, 1)))[:4]),
    trial_ms=("trial_s", lambda x: sorted(set(np.round(x.dropna() * 1000, 1)))[:6]),
    level_min=("level", lambda x: x[x > -99999].min()), level_max=("level", "max"),
    delay_nan=("delay", lambda x: int(x.isna().sum())))
print(summary.to_string())
late = []
for u in units:
    for (_, tr), s in zip(u["trials"].iterrows(), u["spikes"]):
        if len(s) and (s.min() < 0 or s.max() > tr.trial_s + 1e-9):
            late.append(u["path"])
            break
print("\nUnits with a spike outside [0, trial length]:", len(late))
srt = sum(any(np.any(np.diff(s) < 0) for s in u["spikes"]) for u in units)
print("Units with unsorted spike times within a trial:", srt)
dups = U.assign(h=[hash(np.concatenate(u["spikes"] + [np.zeros(1)]).tobytes()) for u in units]).h.duplicated().sum()
print("Units whose spike times duplicate another unit's:", int(dups))

Spike times are in seconds from the start of each trial: tone trials start the tone at 7.75 ms
and last 50 ms, and no spike falls outside its trial's length (taken from the length of the voltage
trace at 48,828.125 Hz, the rate the Methods give). Spike times are sorted within trials, and no
unit's spike train duplicates another's, so no recording was deposited twice. Empty trials hold a
single NaN. The CLICK trials, and the SR, NOISE and CVC trials, carry no delay (NaN): the click's
onset in the trial is not in the files.

The next cell checks that spikes and the trial table share a clock, with a contrast between
conditions: the rate in tone trials minus the rate in the interleaved silent trials, in 5 ms
windows from the start of the trial, over the BF and RLF recordings. If the clock is shared, the
difference is near zero before 7.75 ms and large from a few milliseconds after it. It then checks
that the stored best frequency agrees with the frequency of the highest tone-evoked rate computed
from the spike times.

In [ ]:
edges = np.arange(0, 0.0801, 0.005)
diff_rows = []
for u in units:
    t = u["trials"]
    for exp in ["BF", "RLF"]:
        idx = np.flatnonzero(t.experiment.values == exp)
        if len(idx) == 0:
            continue
        tone = [u["spikes"][i] for i in idx if t.frequency.values[i] != g.SPONT]
        sil = [u["spikes"][i] for i in idx if t.frequency.values[i] == g.SPONT]
        if not tone or not sil:
            continue
        rt = np.histogram(np.concatenate(tone), edges)[0] / len(tone) / 0.005
        rs = np.histogram(np.concatenate(sil), edges)[0] / len(sil) / 0.005
        diff_rows.append(rt - rs)
diff = np.array(diff_rows)
print(f"{len(diff)} BF and RLF recordings with tone and silent trials")
print("Window (ms)  median(tone - silent)  fraction of recordings with tone > silent")
for i in range(len(edges) - 1):
    print(f"{edges[i]*1000:5.0f}-{edges[i+1]*1000:<5.0f}  {np.median(diff[:, i]):10.1f}  {np.mean(diff[:, i] > 0):10.2f}")

# stored BF against the frequency of the highest mean tone-evoked rate in the BF recording
cmp = []
for u in units:
    t, sp = g.trials_of(u, "BF")
    b = U.loc[U.path == u["path"], "bf"].iloc[0]
    if len(t) == 0 or np.isnan(b):
        continue
    t = t.assign(rate=[np.sum((s > a) & (s < a + d)) / d for s, a, d in zip(sp, t.delay, t.duration)])
    m = t[t.frequency != g.SPONT].groupby("frequency").rate.mean()
    cmp.append(np.log2(m.idxmax() / b))
cmp = np.array(cmp)
print(f"\nStored BF against the raw peak of the frequency-response curve, {len(cmp)} units: "
      f"median |log2 ratio| {np.median(np.abs(cmp)):.3f} octaves; within 0.25 octave {np.mean(np.abs(cmp) <= 0.25):.2f}")

Before the tone (0 to 5 ms) the tone and silent trials do not differ; from 10 ms the tone trials
fire far more (median difference 147 spikes/s at 10 to 15 ms, with tone above silent in 96% of
recordings), and after the tone ends (57.75 ms) the tone trials fall below the silent ones, the
offset suppression expected of auditory nerve fibres. The spikes and the trial table share a clock in the BF and RLF recordings, the ones
whose timing matters for the claims (PH and CF use the same delay and duration). The stored BF is
close to the raw peak of the spike-time frequency-response curve (the stored value is the peak of
a smoothing spline, Methods), so the analysis table and the spike times describe the same
recordings.

## table1-animals-units

> The dataset holds 104 experiments (gerbils) with 1160 single auditory nerve fibres; 60
> young-adult (32 female), 13 middle-aged (7 female) and 31 old (11 female) gerbils, with 759, 87
> and 314 single units.

Animals by `subject_id`, age groups by age in days as the code defines them (young <= 365,
middle > 365 and <= 1095, old > 1095).

In [ ]:
A = U.groupby("animal").agg(age_days=("age_days", "first"), sex=("sex", "first"),
                            weight_g=("weight_g", "first"), abr=("abr", "first"),
                            oxygen=("oxygen", "first"), group=("group", "first"), units=("path", "size"))
table = pd.DataFrame({
    "animals": A.groupby("group").size(),
    "females": A[A.sex == "F"].groupby("group").size(),
    "units": U.groupby("group").size()}).loc[["young", "middle", "old"]]
print(table.to_string())
print("Total animals:", len(A), "; total units:", len(U))
paper = {"young": (60, 32, 759), "middle": (13, 7, 87), "old": (31, 11, 314)}
match = all(tuple(table.loc[k]) == v for k, v in paper.items()) and len(A) == 104 and len(U) == 1160
print("Paper: 60 (32) 759; 13 (7) 87; 31 (11) 314; totals 104 and 1160. All eleven counts match:", match)
print("Units per animal: min", A.units.min(), "median", A.units.median(), "max", A.units.max())

**Result: reproduced.** 104 animals and 1160 units; 60 young adults (32 female) with 759 units,
13 middle aged (7 female) with 87, and 31 old (11 female) with 314. All eleven counts match.

## table1-age-abr-weight

> Mean ages of the groups are 5.5, 22.5 and 38.3 months, mean ABR thresholds 18.1, 34.2 and 49.0
> dB SPL, and mean weights 74.6, 83.4 and 86.4 g (young adult, middle aged, old).

Over animals, missing values left out, months = days / (365/12) (check_dataset_metadata.m). The
ABR threshold is the code point of the character in the notes (data check above).

In [ ]:
A["age_months"] = A.age_days / g.DAYS_PER_MONTH
rows = []
expected = {"age_months": (5.5, 22.5, 38.3), "abr": (18.1, 34.2, 49.0), "weight_g": (74.6, 83.4, 86.4)}
for var, exp in expected.items():
    for grp, e in zip(["young", "middle", "old"], exp):
        v = A.loc[A.group == grp, var].dropna()
        rows.append({"variable": var, "group": grp, "n": len(v), "mean": round(v.mean(), 2), "sd": round(v.std(), 2),
                     "min": round(v.min(), 1), "max": round(v.max(), 1), "paper_mean": e,
                     "within_10pct": abs(v.mean() - e) <= 0.1 * e})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("Means within tolerance:", int(res.within_10pct.sum()), "of 9")
print("Animals without an ABR threshold, by group:", A[A.abr.isna()].group.value_counts().to_dict())

**Result: reproduced.** All nine means are within tolerance and equal the paper's after rounding:
ages 5.55, 22.49 and 38.33 months, ABR thresholds 18.10, 34.17 and 49.03 dB SPL, weights 74.61,
83.35 and 86.41 g. The SDs and ranges printed also match Table 1 (for example ABR SD 5.33, 10.19
and 18.00, range 20 to 85 in old animals). The ABR values rest on reading the notes' character as
its code point; 10 young adults and 1 middle-aged animal have "NaN" and are left out, as the
code's `mean(..., 'omitnan')` does.

## table1-bf-range

> Best frequencies range from 428 to 15,825 Hz in young adults, 350 to 15,091 Hz in middle-aged and
> 620 to 15,762 Hz in old gerbils.

A unit's BF is its stored BF, or its CF when it has no BF recording (check_dataset_metadata.m).
The CF is not stored in the files, so it is recomputed from the CF recording (helper,
`cf_from_cf_recording`).

In [ ]:
U["cf"] = [g.cf_from_cf_recording(by_path[p])[0] for p in U.path]
no_bf = U[~U.types.map(lambda x: "BF" in x)]
print("Units without a BF recording:", len(no_bf))
print(no_bf[["animal", "unit", "group", "types", "cf"]].to_string(index=False))
U["bf_or_cf"] = U.bf.where(U.types.map(lambda x: "BF" in x), U.cf)
rng = U.groupby("group").agg(min_bf=("bf_or_cf", "min"), max_bf=("bf_or_cf", "max"), n=("bf_or_cf", "count"),
                             min_stored=("bf", "min"), max_stored=("bf", "max")).loc[["young", "middle", "old"]]
rng["paper"] = ["428-15825", "350-15091", "620-15762"]
print(rng.to_string())
paper = {"young": (428, 15825), "middle": (350, 15091), "old": (620, 15762)}
ok = {k: abs(rng.loc[k, "min_bf"] - lo) <= 0.1 * lo and abs(rng.loc[k, "max_bf"] - hi) <= 0.1 * hi
      for k, (lo, hi) in paper.items()}
print("Within 10% of both bounds:", ok)

**Result: reproduced.** The ranges are 428 to 15,825 Hz (young), 350 to 15,091 Hz (middle aged) and
620 to 15,762 Hz (old), the paper's values exactly, with or without the recomputed CF of the units
that have no BF recording (none of those CFs is outside the range).

## fig3-abr-age

> ABR thresholds rise with age, and hearing loss varies widely among the old animals.

Over animals; reproduced when the mean ABR threshold increases from young to middle aged to old
and the SD of old animals is the largest of the three.

In [ ]:
s = A.groupby("group").abr.agg(["mean", "std", "min", "max", "count"]).loc[["young", "middle", "old"]].round(2)
print(s.to_string())
inc = s["mean"].is_monotonic_increasing
print("Mean increases young < middle < old:", inc, "; old SD the largest:", s["std"].idxmax() == "old")
rho = stats.spearmanr(A.age_days, A.abr, nan_policy="omit")
print(f"Spearman correlation of age with ABR threshold over {A.abr.notna().sum()} animals: "
      f"rho = {rho.statistic:.2f}, p = {rho.pvalue:.2g}")
for sex in ["F", "M"]:
    sub = A[A.sex == sex].groupby("group").abr.mean().loc[["young", "middle", "old"]].round(1)
    print(f"Mean ABR by group, {sex}:", sub.to_dict())
fig, ax = plt.subplots(figsize=(5, 3.5))
for sex, col in [("F", "tab:green"), ("M", "tab:purple")]:
    d = A[A.sex == sex]
    ax.scatter(d.age_days / g.DAYS_PER_MONTH, d.abr, s=18, color=col, label=sex)
ax.set_xlabel("Age (months)"); ax.set_ylabel("ABR threshold (dB SPL)"); ax.legend()
print("Plotted ranges: age", round(A.age_days.min() / g.DAYS_PER_MONTH, 1), "to",
      round(A.age_days.max() / g.DAYS_PER_MONTH, 1), "months; ABR", A.abr.min(), "to", A.abr.max(), "dB SPL")

**Result: reproduced.** Mean ABR thresholds are 18.1, 34.2 and 49.0 dB SPL in young, middle-aged and
old animals, and the SD of the old animals is the largest (18.0, against 5.3 and 10.2), with old
thresholds from 20 to 85 dB SPL. Over 93 animals, age and threshold correlate (Spearman
rho = 0.85, p = 5.7e-27). It rests on reading the ABR character as its code point.

## females-fewer-in-old

> Old gerbils are skewed towards males (11 of 31 female), which the paper attributes to ovarian
> cancer and early death of old females.

In [ ]:
frac = A.groupby("group").apply(lambda d: pd.Series({"female": int((d.sex == "F").sum()), "animals": len(d)}))
frac["fraction_female"] = (frac.female / frac.animals).round(3)
print(frac.loc[["young", "middle", "old"]].to_string())
old_f, young_f = frac.loc["old", "fraction_female"], frac.loc["young", "fraction_female"]
print("Old fraction below one half:", old_f < 0.5, "; below young adults:", old_f < young_f)
f = stats.fisher_exact([[frac.loc["old", "female"], frac.loc["old", "animals"] - frac.loc["old", "female"]],
                        [frac.loc["young", "female"], frac.loc["young", "animals"] - frac.loc["young", "female"]]])
print(f"Fisher's exact test, old against young (not part of the criterion): odds ratio {f.statistic:.2f}, p = {f.pvalue:.3f}")

**Result: reproduced.** 11 of 31 old animals are female (0.355), below one half and below the
young adults (32 of 60, 0.533). The difference from young adults is not significant by Fisher's
exact test (p = 0.125), which the paper does not claim. The cause is not tested.

## oxygen-78

> Of the 104 gerbils, 78 received additional oxygen throughout the experiment.

Counted once per animal from "oxygen: yes|no" in `general/surgery`.

In [ ]:
print(A.oxygen.value_counts().to_string())
print("By group:", A.groupby("group").oxygen.apply(lambda s: int((s == "yes").sum())).to_dict())

**Result: reproduced.** 78 of 104 animals have "oxygen: yes".

## complex-stimuli-experiments

> Responses to paired noise bursts were recorded in 21 of 104 experiments, Schroeder-phase stimuli
> in 5, and consonant-vowel-consonant logatomes in 17.

An experiment counts when at least one of its units has a NOISE, SPS or CVC recording.

In [ ]:
for t in ["NOISE", "SPS", "CVC"]:
    has = U[U.types.map(lambda x: t in x)]
    print(f"{t}: {has.animal.nunique()} experiments, {len(has)} units; by group of animals:",
          has.drop_duplicates("animal").group.value_counts().to_dict())

**Result: reproduced.** NOISE in 21, SPS in 5 and CVC in 17 of the 104 experiments.

## recording-type-counts

> CF recordings were obtained for 119 fibres, CLICK for 261, SR for 203, NOISE for 143, SPS for 22
> and CVC for 135.

Reading 1 (as check_dataset_units.m prints them): CF, units with a non-NaN CF; CLICK, units with
exactly one CLICK recording and a non-NaN 2-bin latency; SR, units with a non-NaN SR from an SR
recording; NOISE, SPS and CVC, units with a recording of that type. Reading 2: units with any
recording of the type. The stored CF and click latency are not in the files; reading 1 uses the
values recomputed with the paper's functions (helper). The click delay is not in the files: the
latency uses 7.75 ms (assumption, helper docstring), with 5 ms shown as a variant.

In [ ]:
U["click_latency"] = [g.click_latency_2bins(by_path[p]) for p in U.path]
U["click_latency_d5"] = [g.click_latency_2bins(by_path[p], delay=0.005) for p in U.path]
n_click_seq = pd.Series([sum(s.split("_")[0] == "CLICK" for s in by_path[p]["sequential_types"]) for p in U.path])
has = {t: U.types.map(lambda x, t=t: t in x) for t in ["CF", "CLICK", "SR", "NOISE", "SPS", "CVC"]}
click_spikes = pd.Series([sum(len(s) for s in g.trials_of(by_path[p], "CLICK")[1]) for p in U.path])
print("CLICK sequential recordings per unit:", n_click_seq[has["CLICK"]].value_counts().to_dict(),
      "; trials per CLICK recording:",
      pd.Series([len(g.trials_of(by_path[p], "CLICK")[0]) for p in U.path[has["CLICK"]]]).value_counts().to_dict())
print("CLICK units with 75 spikes or fewer (not analysed by Clickextract_func.m):", int((has["CLICK"] & (click_spikes <= 75)).sum()))
counts = pd.DataFrame({
    "paper": [119, 261, 203, 143, 22, 135],
    "reading_1": [int(U.cf.notna().sum()), int((has["CLICK"] & (n_click_seq == 1) & U.click_latency.notna()).sum()),
                  int(U.sr_SR.notna().sum()), int(has["NOISE"].sum()), int(has["SPS"].sum()), int(has["CVC"].sum())],
    "reading_2": [int(has[t].sum()) for t in ["CF", "CLICK", "SR", "NOISE", "SPS", "CVC"]],
}, index=["CF", "CLICK", "SR", "NOISE", "SPS", "CVC"])
counts["click_delay_5ms"] = counts.reading_1
counts.loc["CLICK", "click_delay_5ms"] = int((has["CLICK"] & U.click_latency_d5.notna()).sum())
print(counts.to_string())
for r in ["reading_1", "reading_2"]:
    print(f"{r}: {int((counts[r] == counts.paper).sum())} of 6 match")

**Result: partially reproduced.** Five of six counts match under both readings: CF 119, SR 203,
NOISE 143, SPS 22 and CVC 135. CLICK does not: 275 units have a CLICK recording (each one
recording), and 271 of them give a 2-bin latency with the paper's function (272 with a 5 ms
click delay), against 261 in the Fig. 7 legend. The 10 or 14 extra units cannot be traced: the
files carry no stored latency, so it cannot be told whether the paper's latencies were NaN for
them, or whether some of these units had a second CLICK recording in the MATLAB files that the
code would leave out.

## fig7-sampling-by-age

> No SR, SPS or CVC recordings were obtained in middle-aged gerbils, and SPS responses were recorded
> only in young adults.

In [ ]:
tab = pd.DataFrame({t: U[has[t]].group.value_counts() for t in ["CF", "CLICK", "SR", "NOISE", "SPS", "CVC"]}
                   ).reindex(["young", "middle", "old"]).fillna(0).astype(int)
print("Units with a recording of each type, by age group:")
print(tab.to_string())
ok = tab.loc["middle", ["SR", "SPS", "CVC"]].sum() == 0 and tab.loc[["middle", "old"], "SPS"].sum() == 0
print("No middle-aged SR, SPS or CVC, and SPS only in young adults:", ok)

**Result: reproduced.** No middle-aged unit has an SR, SPS or CVC recording, and all 22 SPS units
are from young adults.

## isi-exclusion

> Units with multiple inter-spike intervals below 0.6 ms, the absolute refractoriness of auditory
> nerve fibres, were excluded from the dataset.

Within-trial ISIs (checkAN.m). Readings: (a) any recording of the unit with two or more ISIs
< 0.6 ms; (b) two or more pooled over all the unit's recordings; (c) as (a) for the BF recording
only. A recording here is a recording type of the unit (each type occurs once per unit, data
check).

In [ ]:
isi = {p: g.short_isis(by_path[p]) for p in U.path}
U["short_total"] = [sum(v[0] for v in isi[p].values()) for p in U.path]
U["short_max_rec"] = [max(v[0] for v in isi[p].values()) for p in U.path]
U["short_bf"] = [isi[p].get("BF", (0, 0))[0] for p in U.path]
U["isi_total"] = [sum(v[1] for v in isi[p].values()) for p in U.path]
fail = {"a": int((U.short_max_rec >= 2).sum()), "b": int((U.short_total >= 2).sum()), "c": int((U.short_bf >= 2).sum())}
print("Units failing (two or more ISIs < 0.6 ms):", fail, "of", len(U))
print("Units with at least one ISI < 0.6 ms:", int((U.short_total >= 1).sum()))
print("\nDistribution of a unit's count of short ISIs (pooled):")
print(U.short_total.value_counts().sort_index().pipe(lambda s: pd.concat([s[s.index <= 10], pd.Series({">10": s[s.index > 10].sum()})])).to_string())
frac = U.short_total / U.isi_total.where(U.isi_total > 0)
print(f"\nShort ISIs as a fraction of all ISIs, among failing units (b): median {frac[U.short_total >= 2].median():.4f}, "
      f"max {frac[U.short_total >= 2].max():.3f}")
print("Failing units (a) by age group:", U[U.short_max_rec >= 2].group.value_counts().to_dict(),
      "; by animal: in", U[U.short_max_rec >= 2].animal.nunique(), "of", U.animal.nunique(), "animals")
short = np.concatenate([np.diff(s)[np.diff(s) < g.SHORT_ISI_S] for u in units for s in u["spikes"] if len(s) > 1])
print(f"\nAll short ISIs: {len(short)}; in ms, quantiles 0, 0.25, 0.5, 0.75, 1:",
      np.round(np.quantile(short * 1000, [0, 0.25, 0.5, 0.75, 1]), 3).tolist())
print("Of them below 0.1 ms:", int((short < 0.0001).sum()), "; 0.1 to 0.3 ms:", int(((short >= 0.0001) & (short < 0.0003)).sum()),
      "; 0.3 to 0.6 ms:", int((short >= 0.0003).sum()))
for lim in [0.0003, 0.0001]:
    n = sum(any(sum(int(np.sum(np.diff(s) < lim)) for s in g.trials_of(by_path[p], e)[1]) >= 2
                for e in by_path[p]["trials"].experiment.unique()) for p in U.path)
    print(f"Variant limit {lim*1000:.1f} ms, reading (a): {n} units fail")

**Result: not reproduced.** Units with two or more within-trial ISIs below 0.6 ms are in the
dataset under every reading: 370 of 1160 under (a), 391 under (b) and 148 under (c), from 79 of
the 104 animals; 538 units have at least one. In the failing units the short ISIs are a small
fraction of all ISIs (median 0.0013, largest 0.223), and 162 units have more than 10. Of the
13,904 short ISIs, 10,192 are between 0.3 and 0.6 ms and 1171 below 0.1 ms. With a limit of
0.3 ms, 140 units still fail under (a); with 0.1 ms, 5 do. The paper's check was visual and does
not say what "multiple" means; under the three readings the claims file names, the dataset holds
units the stated rule would exclude.

## no-prepotentials

> No median spike waveform showed a prepotential, which would mark a ventral cochlear nucleus bushy
> cell.

Blocked: methods underspecified. The check was a visual inspection of the median spike
waveform (checkAN.m); neither the paper nor its code gives a rule for a prepotential, so no
criterion fixed from the paper can judge it (claims file). The raw voltage traces are in the files
(`acquisition`), so the waveforms could be inspected; they are not read here.

## rlf-monotonic

> Rate-level functions were checked for atypical shapes ..., and units whose rate-level function was
> nonmonotonic at levels below 80 dB SPL were excluded.

Blocked: methods underspecified. The check was visual and the paper gives no rule for
nonmonotonicity. As a description only (claims file): the stored rates and SDs are not in the
files, so the rate per level is computed from the spike times in the stimulus window (helper,
`rlf_rates`), and the units are counted whose mean rate falls between successive levels below
80 dB SPL by more than the SD at the lower level.

In [ ]:
drops, n_rlf = [], 0
for p in U.path[U.types.map(lambda x: "RLF" in x)]:
    r = g.rlf_rates(by_path[p])
    if r is None or len(r) < 2:
        continue
    n_rlf += 1
    r = r[r.index < 80]
    d = r["mean"].diff().shift(-1)            # next level minus this level
    drops.append(int(((-d) > r["std"]).sum()))
drops = np.array(drops)
print(f"RLF recordings: {n_rlf}; with a drop larger than the SD below 80 dB SPL: {int((drops >= 1).sum())} "
      f"({np.mean(drops >= 1):.2f}); with two or more such drops: {int((drops >= 2).sum())}")

## fig5b-example

> The example fibre of Fig. 5b (animal G220908, unit 3p_181) had a best frequency of 7.6 kHz and a
> threshold of 19 dB SPL.

In [ ]:
row = U[(U.animal == "G220908") & (U.unit == "3p_181")]
print(row[["path", "group", "bf", "rlf_threshold", "sr_best", "types"]].to_string(index=False))
u = by_path[row.path.iloc[0]]
t = u["trials"]
bf_t = t[(t.experiment == "BF") & (t.frequency != g.SPONT)]
rlf_t = t[(t.experiment == "RLF") & (t.frequency != g.SPONT)]
print("BF recording: levels (dB SPL)", sorted(np.round(bf_t.level.unique(), 1)), "; repetitions per frequency",
      sorted(bf_t.groupby("frequency").size().unique()))
print("RLF recording: frequency", sorted(rlf_t.frequency.unique()), "; levels", sorted(np.round(rlf_t.level.unique(), 1)),
      "; repetitions per level", sorted(rlf_t.groupby("level").size().unique()))
print("BF within 10% of 7.6 kHz:", abs(row.bf.iloc[0] / 1000 - 7.6) <= 0.76,
      "; threshold within 10% of 19 dB:", abs(row.rlf_threshold.iloc[0] - 19) <= 1.9)

**Result: reproduced.** The unit's stored BF is 7620 Hz and its RLF threshold 19 dB SPL. As the
legend says, its BF recording is at 30 dB SPL with 5 repetitions per frequency, and its RLF
recording (at 7700 Hz) has 10 repetitions per level and includes 40 dB SPL.

## fig6a-young-two-sensitive-regions

> Fibres of young-adult gerbils show two regions of best sensitivity separated by a region around
> 3 to 4 kHz with higher thresholds and fewer fibres; the highest thresholds are at 3.5 kHz.

Young-adult units with a stored BF and RLF threshold; mean threshold in 24 logarithmic BF bins
from 0.3 to 20 kHz (a bin includes its upper edge), each plotted at its upper edge
(check_dataset_units.m). Readings: (a) the largest bin mean over all bins; (b) the largest bin
mean between the two sensitive regions, a local maximum with lower bin means on each side.

In [ ]:
y = U[U.group == "young"]
e = np.logspace(np.log10(0.3), np.log10(20), 25)
k = y.bf / 1000
bins = []
for i in range(24):
    m = (k > e[i]) & (k <= e[i + 1])
    bins.append({"lower_khz": round(e[i], 2), "plotted_at_khz": round(e[i + 1], 2), "units": int(m.sum()),
                 "with_threshold": int(y.rlf_threshold[m].notna().sum()), "mean_threshold": y.rlf_threshold[m].mean()})
B6a = pd.DataFrame(bins)
print(B6a.round({"mean_threshold": 1}).to_string(index=False))
filled = B6a[B6a.mean_threshold.notna()].reset_index(drop=True)
a_top = filled.loc[filled.mean_threshold.idxmax()]
print(f"\nReading (a): highest bin mean {a_top.mean_threshold:.1f} dB at {a_top.plotted_at_khz} kHz (n = {a_top.with_threshold})")
# the two sensitive regions: the lowest bin mean below 3.48 kHz and the lowest above it
low = filled[filled.plotted_at_khz < 3.48]; high = filled[filled.plotted_at_khz > 3.48]
lo_min, hi_min = low.mean_threshold.idxmin(), high.mean_threshold.idxmin()
mid = filled.loc[lo_min + 1:hi_min - 1]
b_top = mid.mean_threshold.idxmax()
print(f"Sensitive regions: lowest means {filled.mean_threshold[lo_min]:.1f} dB at {filled.plotted_at_khz[lo_min]} kHz and "
      f"{filled.mean_threshold[hi_min]:.1f} dB at {filled.plotted_at_khz[hi_min]} kHz")
print(f"Reading (b): highest mean between them {filled.mean_threshold[b_top]:.1f} dB at {filled.plotted_at_khz[b_top]} kHz; "
      f"neighbours {filled.mean_threshold[b_top-1]:.1f} and {filled.mean_threshold[b_top+1]:.1f} dB")
print("Units per bin around it (2.06 to 4.93 kHz):", B6a[(B6a.plotted_at_khz > 2) & (B6a.plotted_at_khz < 5)].units.tolist())
fig, ax = plt.subplots(figsize=(5, 3.5))
for grp, col in [("young", "b"), ("middle", "orange"), ("old", "r")]:
    d = U[U.group == grp]
    ax.scatter(d.bf / 1000, d.rlf_threshold, s=6, color=col, alpha=0.5, label=grp)
ax.plot(B6a.plotted_at_khz, B6a.mean_threshold, "b-")
ax.set_xscale("log"); ax.set_xlim(0.3, 20); ax.set_ylim(0, 100)
ax.set_xlabel("Best frequency (kHz)"); ax.set_ylabel("Threshold (dB SPL)"); ax.legend(fontsize=7)
out = U[(U.rlf_threshold.notna()) & ((U.bf / 1000 < 0.3) | (U.bf / 1000 > 20) | (U.rlf_threshold > 100) | (U.rlf_threshold < 0))]
print("Points outside the axes:", len(out), "; thresholds range", round(U.rlf_threshold.min(), 1), "to", round(U.rlf_threshold.max(), 1))

**Result: reproduced.** Under reading (b) the young-adult mean threshold between the two sensitive
regions is highest in the bin plotted at 3.48 kHz (27.6 dB SPL), with lower means on both sides
(25.4 and 24.4 dB), and the two regions of best sensitivity have their lowest means at 1.73 kHz
(20.9 dB) and 8.34 kHz (21.2 dB). Under reading (a) the highest bin mean of all is 36.7 dB at
0.72 kHz, from 3 units. The bins plotted at 2.06 to 4.14 kHz hold 30 to 34 units, fewer than those
plotted at 4.93 to 11.83 kHz (43 to 83) but about as many as those plotted at 1.02 to 1.73 kHz
(33 to 36).

## fig6b-young-bf-gap

> Among young-adult fibres, the fewest best frequencies fall in the bin bordered by 2.5 and 3.0 kHz.

Histogram of young-adult stored BFs, edges logspace(log10(200), log10(20000), 25) Hz; the named bin
is 2423 to 2936 Hz. Readings: (a) fewest of all bins between the lowest and highest non-empty
bins (ties allowed); (b) fewer units than both neighbours.

In [ ]:
e = np.logspace(np.log10(200), np.log10(20000), 25)
h, _ = np.histogram(y.bf.dropna(), e)
H = pd.DataFrame({"lower_hz": np.round(e[:-1]).astype(int), "upper_hz": np.round(e[1:]).astype(int), "units": h})
print(H.to_string(index=False))
i = int(np.flatnonzero(H.lower_hz == 2423)[0])
nz = np.flatnonzero(h)
span = h[nz[0]:nz[-1] + 1]
print(f"\nBin 2423-2936 Hz: {h[i]} units; neighbours {h[i-1]} and {h[i+1]}")
print(f"Reading (a): fewest in the non-empty span is {span.min()} (bin {H.lower_hz[nz[0] + span.argmin()]} Hz); holds: {h[i] == span.min()}")
print(f"Reading (b): fewer than both neighbours: {h[i] < h[i-1] and h[i] < h[i+1]}")
mid = H[(H.lower_hz >= 1363) & (H.upper_hz <= 4309)]
print("Bins from 1363 to 4309 Hz:", mid.units.tolist(), "; the fewest there:", mid.units.min(),
      "in bins", mid[mid.units == mid.units.min()].lower_hz.tolist())
print("Total young units with a BF:", int(h.sum()), "; plotted range", int(y.bf.min()), "to", int(y.bf.max()), "Hz")

**Result: not reproduced.** The bin from 2423 to 2936 Hz holds 34 young-adult units. It is not the
fewest of the non-empty span (the lowest bin, 356 Hz, holds 1, and the bins below 766 Hz hold 1 to
11), and it is not a local minimum: its neighbours hold 34 and 35. Among the bins from 1363 to
4309 Hz it ties for the fewest (34) with two others, so the dip in the histogram is shallow; with
the code's bins the stated bin is not the one with the fewest fibres.

## fig6c-sr-distributions

> Among high-frequency fibres (BF above 3.5 kHz) there is a cluster with low spontaneous rates,
> and low-frequency fibres have a bimodal distribution of spontaneous rate with modes near 5 and 60
> spikes/s.

Young-adult units, SR as the code's best estimate, split at BF 3.5 kHz, histogram of 5 spikes/s
bins from 0 to 200. A mode is a local maximum (more units than both neighbours, or the first bin
when it exceeds the second); the two largest are taken, separated by a bin with fewer units than
both.

In [ ]:
def local_maxima(h):
    out = []
    for i in range(len(h)):
        left = h[i - 1] if i > 0 else -1
        right = h[i + 1] if i < len(h) - 1 else -1
        if h[i] > left and h[i] > right:
            out.append(i)
    return out

edges = np.arange(0, 205, 5)
res6c = {}
for name, m in [("low BF (<= 3.5 kHz)", y.bf <= 3500), ("high BF (> 3.5 kHz)", y.bf > 3500)]:
    h, _ = np.histogram(y.sr_best[m].dropna(), edges)
    lm = local_maxima(h)
    top2 = sorted(sorted(lm, key=lambda i: -h[i])[:2])
    res6c[name] = (h, lm, top2)
    print(f"{name}: {int(m.sum())} units, {int(y.sr_best[m].notna().sum())} with an SR; counts per 5 spikes/s bin (0 to 150):")
    print("  ", h[:30].tolist())
    print("   local maxima (bin start, count):", [(int(edges[i]), int(h[i])) for i in lm])
    print("   two largest:", [(int(edges[i]), int(h[i])) for i in top2],
          "; fewest between them:", int(h[top2[0]:top2[1] + 1].min()) if len(top2) == 2 else None)
h, lm, top2 = res6c["low BF (<= 3.5 kHz)"]
low_ok = len(top2) == 2 and edges[top2[0]] in (0, 5) and 50 <= edges[top2[1]] and edges[top2[1] + 1] <= 70 \
    and h[top2[0]:top2[1] + 1].min() < min(h[top2[0]], h[top2[1]])
hh, lmh, _ = res6c["high BF (> 3.5 kHz)"]
largest_high = max(lmh, key=lambda i: hh[i])
high_ok = edges[largest_high + 1] <= 18
print("\nLow BF bimodal with modes in 0-10 and 50-70 spikes/s:", low_ok)
print(f"High BF largest local maximum at {int(edges[largest_high])}-{int(edges[largest_high+1])} spikes/s, below 18: {high_ok}")
print("Fraction below 18 spikes/s: low BF", round(float((y.sr_best[y.bf <= 3500] < 18).mean()), 3),
      "; high BF", round(float((y.sr_best[y.bf > 3500] < 18).mean()), 3))
print("SR range (young):", y.sr_best.min(), "to", round(y.sr_best.max(), 1), "; above 200:", int((y.sr_best > 200).sum()))

**Result: reproduced.** In low-BF young-adult units the two largest modes of spontaneous rate are
the bins 0 to 5 spikes/s (27 units) and 55 to 60 spikes/s (21 units), separated by bins of fewer
units; both lie in the bins the criterion allows. In high-BF units the largest local maximum is
the 0 to 5 spikes/s bin (66 units), below the 18 spikes/s cutoff.

## fig6d-max-phase-locking-bf

> The highest best frequency at which significant phase locking was recorded in young-adult gerbils
> was 4.6 kHz.

Vector strength and its p-value per level of the PH recording, recomputed from the spike times
with PHextract_func.m and TytoView_calcVS.m (helper); significant when p is not NaN and < 0.001.

In [ ]:
U["max_vs"] = [g.max_significant_vs(by_path[p]) for p in U.path]
ph = U[U.types.map(lambda x: "PH" in x)]
nfreq = pd.Series([g.vector_strength_by_level(by_path[p]).n_freqs.max() for p in ph.path])
print("Units with a PH recording:", len(ph), "; PH recordings with more than one tone frequency:", int((nfreq > 1).sum()))
print("With a significant vector strength:", int(ph.max_vs.notna().sum()), "; by group:",
      ph[ph.max_vs.notna()].group.value_counts().to_dict())
sig = ph[ph.max_vs.notna() & ph.bf.notna()]
for grp in ["young", "middle", "old"]:
    s = sig[sig.group == grp].sort_values("bf")
    print(f"{grp}: highest BF with significant phase locking {s.bf.max():.0f} Hz; next highest {s.bf.iloc[-4:-1].tolist()}")
ys = ph[ph.group == "young"]
print("Young PH units with BF above 4.6 kHz:", ys[ys.bf > 4600][["bf", "max_vs"]].to_dict("records"))
top = sig[sig.group == "young"].bf.max() / 1000
print(f"Within 10% of 4.6 kHz: {abs(top - 4.6) <= 0.46}")
for p_lim in [0.01, 0.0001]:
    alt = []
    for p in ph.path[ph.group == "young"]:
        v = g.vector_strength_by_level(by_path[p])
        if ((v.p < p_lim) & v.p.notna()).any():
            alt.append(U.loc[U.path == p, "bf"].iloc[0])
    print(f"Variant p < {p_lim}: highest young BF {np.nanmax(alt):.0f} Hz")

**Result: reproduced.** With vector strength recomputed by the paper's function, the highest BF of a
young-adult unit with significant phase locking (p < 0.001, the Methods' threshold) is 4598 Hz,
4.6 kHz; the next highest are 4121 to 4377 Hz. The value depends on the threshold: with p < 0.01
it is 6539 Hz, with p < 0.0001 it is 4377 Hz. No young unit with a higher BF (27 PH units from
4718 to 12,359 Hz) reaches p < 0.001.

## fig6d-highest-vs-low-bf-low-sr

> The highest vector strengths are found among fibres with low best frequencies (below 1.5 kHz) and
> low spontaneous rates (below 18 spikes/s).

Maximum significant vector strength per unit, SR best estimate, stored BF. Readings, for young
adults and for all ages: (a) the unit with the largest maximum vector strength has BF < 1.5 kHz
and SR < 18; (b) of the four groups the two cutoffs form, the median is highest in low-BF low-SR.

In [ ]:
res = []
for name, s in [("young", sig[sig.group == "young"]), ("all ages", sig)]:
    grp4 = np.where(s.bf < 1500, "BF<1.5k", "BF>=1.5k") + " / " + np.where(s.sr_best < 18, "SR<18", "SR>=18")
    tab = s.groupby(grp4).max_vs.agg(["median", "max", "size"]).round(3)
    topu = s.loc[s.max_vs.idxmax()]
    ra = bool(topu.bf < 1500 and topu.sr_best < 18)
    rb = tab["median"].idxmax() == "BF<1.5k / SR<18"
    res.append((name, ra, rb))
    print(f"{name}:\n{tab.to_string()}")
    print(f"  largest: vs {topu.max_vs:.3f}, BF {topu.bf:.0f} Hz, SR {topu.sr_best} spikes/s; reading (a) holds: {ra}; reading (b) holds: {rb}")
    print("  five largest:", s.nlargest(5, "max_vs")[["bf", "sr_best", "max_vs"]].round(3).to_dict("records"))
low = sig[(sig.group == "young")]
lo = low[(low.bf < 1500) & (low.sr_best < 18)].max_vs; hi = low[(low.bf < 1500) & (low.sr_best >= 18)].max_vs
print(f"Young, BF < 1.5 kHz: low-SR against high-SR, Mann-Whitney p = {stats.mannwhitneyu(lo, hi).pvalue:.3g} "
      f"(n {len(lo)} and {len(hi)}; not part of the criterion)")
print("All readings hold:", all(a and b for _, a, b in res))

**Result: reproduced.** Under all four readings the highest vector strengths are in low-BF,
low-SR fibres: the largest maximum vector strength (0.882) is from a unit with BF 428 Hz and SR
1.25 spikes/s, and the median is highest in the low-BF low-SR group (0.821 in 6 young-adult
units, 0.804 in 19 over all ages), above low-BF high-SR (0.695 and 0.702). The low-SR group is
small; the comparison of its values with the low-BF high-SR group is printed above.

## fig7a-cf-matches-bf

> Characteristic frequency from the CF recordings correlates strongly with best frequency from the
> BF recordings, with no systematic deviation towards higher or lower frequencies.

CF recomputed from the CF recording (helper; the automatic first pass of CFextract_func.m, without
the experimenter's manual corrections, which are not in the files); BF stored. Pearson correlation
of log CF with log BF (Spearman as a variant), median log2(CF/BF) with a 95% bootstrap interval
(10,000 resamples, seed 1).

In [ ]:
rng = np.random.default_rng(1)
c = U[U.cf.notna() & U.bf.notna()]
pr = stats.pearsonr(np.log(c.cf), np.log(c.bf)); sp = stats.spearmanr(c.cf, c.bf)
l2 = np.log2(c.cf / c.bf).values
boot = np.median(rng.choice(l2, size=(10000, len(l2))), axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Units with a CF and a BF: {len(c)} (CF units {int(has['CF'].sum())}, of which without BF {int((has['CF'] & U.bf.isna()).sum())})")
print(f"Pearson r(log CF, log BF) = {pr.statistic:.3f}, p = {pr.pvalue:.2g}; Spearman rho = {sp.statistic:.3f}, p = {sp.pvalue:.2g}")
print(f"Median log2(CF/BF) = {np.median(l2):.4f} (95% CI {lo:.4f} to {hi:.4f}); median ratio {2**np.median(l2):.4f}")
print(f"Units with CF above BF: {int((l2 > 0).sum())}, below: {int((l2 < 0).sum())}, equal: {int((l2 == 0).sum())}")
print(f"|log2(CF/BF)| > 0.5 octave: {int((np.abs(l2) > 0.5).sum())}")
print("By group:", c.assign(l2=l2).groupby("group").l2.agg(["median", "size"]).round(3).to_dict("index"))
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(c.bf / 1000, c.cf / 1000, s=10)
ax.plot([0.3, 20], [0.3, 20], "k--"); ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("BF (kHz)"); ax.set_ylabel("CF, recomputed (kHz)")
print("Plotted ranges: BF", c.bf.min(), "to", c.bf.max(), "Hz; CF", c.cf.min(), "to", c.cf.max(), "Hz")

**Result: reproduced.** Over 114 units with a recomputed CF and a stored BF, log CF and log BF
correlate with r = 0.973 (p = 2.1e-73; Spearman 0.964), and the median CF/BF ratio is 1.003
(log2 0.0041, 95% interval -0.0111 to 0.0398), within 10% of 1; 58 units have CF above BF and 56
below, and 13 differ by more than half an octave. The CF is the automatic first pass of the
paper's function; the experimenter's manual corrections are not in the files.

## fig7b-click-latency-bf

> Click latency is strongly negatively correlated with best frequency, reflecting the travelling
> wave delay along the cochlea.

Units with one CLICK recording, a 2-bin latency recomputed with the paper's function, and a
stored BF; Pearson correlation of latency with log BF, Spearman as a variant. The click delay is
not in the files; the main reading assumes 7.75 ms, the delay of every tone recording, and 5 ms is
shown as a variant.

In [ ]:
for col, label in [("click_latency", "delay 7.75 ms"), ("click_latency_d5", "delay 5 ms")]:
    k = U[U[col].notna() & U.bf.notna()]
    pr = stats.pearsonr(k[col], np.log(k.bf)); sp = stats.spearmanr(k[col], k.bf)
    print(f"{label}: {len(k)} units; latency {k[col].min()*1000:.2f} to {k[col].max()*1000:.2f} ms "
          f"(median {k[col].median()*1000:.2f}); Pearson r = {pr.statistic:.3f}, p = {pr.pvalue:.2g}; "
          f"Spearman rho = {sp.statistic:.3f}, p = {sp.pvalue:.2g}")
k = U[U.click_latency.notna() & U.bf.notna()]
print("Median latency (ms) by BF band:",
      k.groupby(pd.cut(k.bf, [0, 1000, 2000, 4000, 8000, 20000])).click_latency.median().mul(1000).round(2).to_dict())
print("By age group, Pearson r:", {grp: round(stats.pearsonr(d.click_latency, np.log(d.bf)).statistic, 3)
                                  for grp, d in k.groupby("group") if len(d) > 3})
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(k.bf / 1000, k.click_latency * 1000, s=8)
ax.set_xscale("log"); ax.set_xlim(0.3, 20); ax.set_ylim(0, 3.5)
ax.set_xlabel("BF (kHz)"); ax.set_ylabel("Click latency, 2-bin (ms)")
print("Points outside the axes:", int(((k.click_latency * 1000 > 3.5) | (k.bf < 300) | (k.bf > 20000)).sum()))

**Result: reproduced.** Over 268 units the recomputed 2-bin click latency falls with BF: Pearson
r = -0.882 with log BF (p = 5.9e-89), Spearman -0.864, and in each age group (-0.78 to -0.92);
the median latency falls from 2.09 ms below 1 kHz to 0.73 ms above 8 kHz. With a 5 ms click delay
the correlations are -0.662 and -0.820 over 269 units, both negative with p < 0.05. With 7.75 ms the latencies span 0.42 to 2.78
ms, inside the 0 to 3.5 ms axis of Fig. 7b, which is consistent with the assumed delay.

## fig7ab-representative-sampling

> The best-frequency distributions of the units with CF and with CLICK recordings are
> representative of the distribution of the full dataset.

Reference (i) young-adult BFs or (ii) all units' BFs; comparison (a) two-sample KS test of log BF,
no difference at p < 0.05, or (b) the subset's median BF within 10% of the reference's. Stored
BFs. The CF subset is the units with a CF recording; the CLICK subset is the units with a
recomputed latency (reading 1 of recording-type-counts), with all CLICK units as a variant.

In [ ]:
refs = {"(i) young": np.log(U[U.group == "young"].bf.dropna()), "(ii) all": np.log(U.bf.dropna())}
subsets = {"CF": U[has["CF"]].bf.dropna(), "CLICK": U[U.click_latency.notna()].bf.dropna(),
           "CLICK, any recording": U[has["CLICK"]].bf.dropna()}
rows = []
for sname, s in subsets.items():
    for rname, r in refs.items():
        ks = stats.ks_2samp(np.log(s), r)
        ratio = np.exp(np.median(np.log(s))) / np.exp(np.median(r))
        rows.append({"subset": sname, "n": len(s), "reference": rname, "ks_D": round(ks.statistic, 3),
                     "ks_p": round(ks.pvalue, 4), "(a) holds": ks.pvalue >= 0.05,
                     "median_subset_hz": round(float(np.exp(np.median(np.log(s))))),
                     "median_ref_hz": round(float(np.exp(np.median(r)))), "ratio": round(ratio, 3),
                     "(b) holds": abs(ratio - 1) <= 0.1})
R = pd.DataFrame(rows)
print(R.to_string(index=False))
for reading in ["(a) holds", "(b) holds"]:
    for rname in refs:
        both = R[(R.reference == rname) & R.subset.isin(["CF", "CLICK"])][reading].all()
        print(f"Reading {reading[:3]} with reference {rname}: holds for both CF and CLICK: {both}")
print("\nCF units by age group:", U[has["CF"]].group.value_counts().to_dict(),
      "; BF quartiles (Hz) CF", np.percentile(subsets["CF"], [25, 50, 75]).round().tolist(),
      "all", np.percentile(U.bf.dropna(), [25, 50, 75]).round().tolist())

**Result: not reproduced.** No reading holds for both subsets. The CF units' BFs differ from the
reference under both readings: KS p = 0.018 against young adults and 0.003 against all units, and
their median BF (3454 Hz) is 0.75 times the reference's (4589 and 4592 Hz). The CLICK units'
BFs do not differ by the KS test (p = 0.20 and 0.45), but their median (5180 Hz) is 1.13 times
the reference's, outside 10%. The CF part fails every reading, so the outcome does not depend on
the choice among them; whether the paper meant a visual likeness that a shift of this size would
not contradict cannot be told from its text.

## old-success-rate

> Successful single-fibre recordings were obtained in 43% of old gerbils that were tried, against
> 88% of young adults.

Blocked: data missing. The denominators are the animals operated on, including those that gave
no single units. The dataset holds only animals with recorded units (every one of the 104 subjects
has at least one unit file, data check), and neither the files nor the dandiset's metadata list
the animals that gave none.

In [ ]:
print("Animals in the dataset with no unit file:", int((A.units == 0).sum()), "; animals:", len(A))
print("Dandiset description mentions unsuccessful experiments:",
      any(w in meta["description"].lower() for w in ["unsuccessful", "success", "failed"]))

## psth-primary-like

> Responses to tones at 20 and 30 dB above threshold were examined for non-primary-like shapes ...
> and units with clearly non-primary-like responses were excluded.

Blocked: methods underspecified. The check was visual; the paper and its code give no rule
for a non-primary-like shape (claims file).

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| table1-animals-units | reproduced |
| table1-age-abr-weight | reproduced (ABR read as character code) |
| table1-bf-range | reproduced |
| fig3-abr-age | reproduced (ABR read as character code) |
| females-fewer-in-old | reproduced |
| oxygen-78 | reproduced |
| complex-stimuli-experiments | reproduced |
| recording-type-counts | partially reproduced (CLICK 271 or 275, not 261) |
| fig7-sampling-by-age | reproduced |
| isi-exclusion | not reproduced |
| no-prepotentials | blocked: methods underspecified |
| rlf-monotonic | blocked: methods underspecified |
| fig5b-example | reproduced |
| fig6a-young-two-sensitive-regions | reproduced (reading b) |
| fig6b-young-bf-gap | not reproduced |
| fig6c-sr-distributions | reproduced |
| fig6d-max-phase-locking-bf | reproduced (recomputed vector strength) |
| fig6d-highest-vs-low-bf-low-sr | reproduced (recomputed vector strength) |
| fig7a-cf-matches-bf | reproduced (recomputed CF) |
| fig7b-click-latency-bf | reproduced (recomputed latency, assumed click delay) |
| fig7ab-representative-sampling | not reproduced |
| old-success-rate | blocked: data missing |
| psth-primary-like | blocked: methods underspecified |

How the analysis differs from the paper's. The paper's scripts read stored analysed outcomes
from the MATLAB files. The NWB files keep only the best frequency, spontaneous rate and threshold
per recording, so the characteristic frequency, vector strength and click latency are recomputed
from the spike times with the paper's functions, without the experimenter's manual corrections
for the CF. The click delay is not in the files and is assumed to be 7.75 ms. The ABR threshold is
read from a character code.

What was found about the dataset. All 1160 units and 104 animals of Table 1 are there, with
ages, sexes, weights, oxygen, and the recording types of Fig. 7, and every count and mean of
Table 1 matches. The ABR threshold is stored as a single character whose code point is the
threshold. The CF, vector strength with its p-value, and click latencies that the paper describes
as stored outcomes are not in the NWB files, and the click delay is missing from the trial table.
Many units have within-trial inter-spike intervals below 0.6 ms, often below 0.1 ms. The archive's
summary lists the recordings as voltage clamp, while they are extracellular recordings of
single fibres.